In [3]:
"""
API Status Prediction Model using XGBoost.

This script processes structured log data, aggregates HTTP status codes into 
binary categories (SUCCESS vs. ERROR), and trains an XGBoost binary classifier.
Evaluation metrics include LogLoss, classification reports, and AUC-ROC scores.
"""

import gc
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ==========================================
# CONFIGURATION
# ==========================================
DATA_PATH = "transformed_data_sampled.csv"
TARGET_COL = "status"

# ==========================================
# 1. DATA INGESTION
# ==========================================
print("1. Loading data...")
sample = pd.read_csv(DATA_PATH, nrows=100)

dtypes = {}
for col in sample.columns:
    if sample[col].dtype == "float64":
        dtypes[col] = "float32"
    elif sample[col].dtype == "int64":
        dtypes[col] = "int32"
    else:
        dtypes[col] = sample[col].dtype

df = pd.read_csv(DATA_PATH, dtype=dtypes)

# ==========================================
# 2. STATUS CODE AGGREGATION
# ==========================================
print("2. Aggregating status codes...")
status_group_mapping = {
    0: "SUCCESS",        # 200
    1: "SUCCESS",        # 201
    2: "SUCCESS",        # 204
    3: "ERROR",          # 400
    4: "ERROR",          # 401
    5: "ERROR",          # 404
    6: "ERROR"           # 500
}

df["status_grouped"] = df[TARGET_COL].map(status_group_mapping)

le = LabelEncoder()
df[TARGET_COL] = le.fit_transform(df["status_grouped"])
df.drop(columns=["status_grouped"], inplace=True)

# ==========================================
# 3. FEATURE / TARGET SPLIT
# ==========================================
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]
del df
gc.collect()

# ==========================================
# 4. TRAIN / VALIDATION SPLIT
# ==========================================
print("\n3. Splitting dataset...")
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

num_classes = len(np.unique(y_train))
del X, y
gc.collect()

# ==========================================
# 5. CLASS WEIGHTS BEREKENEN
# ==========================================
print("\n4. Calculating class weights...")
class_counts = y_train.value_counts().sort_index()
class_weights = {
    cls: len(y_train) / (len(class_counts) * count)
    for cls, count in class_counts.items()
}

sample_weights = y_train.map(class_weights).values

# ==========================================
# 6. DMATRIX CONVERSION
# ==========================================
dtrain = xgb.DMatrix(X_train, label=y_train, weight=sample_weights)
dval = xgb.DMatrix(X_val, label=y_val)

# ==========================================
# 7. HYPERPARAMETERS (with AUC evaluation)
# ==========================================
params = {
    "objective": "binary:logistic", 
    "tree_method": "hist",
    "max_depth": 6,
    "min_child_weight": 5,
    "learning_rate": 0.05,
    
    # Multiple evaluation metrics specified as a list
    "eval_metric": ["logloss", "auc"], 

    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "max_delta_step": 1,
    "verbosity": 1
}

# ==========================================
# 8. MODEL TRAINING
# ==========================================
print("\n5. Starting model training...")
evallist = [(dval, "validation"), (dtrain, "train")]
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    evals=evallist,            
    early_stopping_rounds=50,
    verbose_eval=50
)

# ==========================================
# 9. EVALUATION (including AUC-ROC score)
# ==========================================
print("\n=== EVALUATION ===")
best_iteration = bst.best_iteration
preds_prob = bst.predict(dval, iteration_range=(0, best_iteration + 1))

# Binary predictions (0 or 1) using a customized threshold of 0.6
y_pred = (preds_prob >= 0.6).astype(int)

# Calculating AUC-ROC Score using continuous probabilities
auc_score = roc_auc_score(y_val, preds_prob)
print(f"Final AUC-ROC Score: {auc_score:.4f}")
print("-" * 40)

print("\nClassification Report:")
print(classification_report(y_val, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

print("\nLabel mapping:")
for i, cls in enumerate(le.classes_):
    print(f"{i} -> {cls}")

1. Data inladen...
2. Statuscodes groeperen...

3. Dataset opsplitsen...


/tmp/ipykernel_1365466/3876446727.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["status_grouped"] = df[TARGET_COL].map(status_group_mapping)



4. Class weights berekenen...

5. Start training...
[0]	validation-logloss:0.68423	validation-auc:0.80150	train-logloss:0.68194	train-auc:0.80017
[50]	validation-logloss:0.51115	validation-auc:0.81836	train-logloss:0.48596	train-auc:0.81807
[100]	validation-logloss:0.48904	validation-auc:0.82579	train-logloss:0.46408	train-auc:0.82600
[150]	validation-logloss:0.48261	validation-auc:0.83127	train-logloss:0.45764	train-auc:0.83199
[200]	validation-logloss:0.47780	validation-auc:0.83603	train-logloss:0.45290	train-auc:0.83766
[250]	validation-logloss:0.47379	validation-auc:0.84004	train-logloss:0.44858	train-auc:0.84253
[300]	validation-logloss:0.47033	validation-auc:0.84340	train-logloss:0.44549	train-auc:0.84595
[350]	validation-logloss:0.46806	validation-auc:0.84534	train-logloss:0.44297	train-auc:0.84825
[400]	validation-logloss:0.46580	validation-auc:0.84707	train-logloss:0.44081	train-auc:0.85008
[450]	validation-logloss:0.46474	validation-auc:0.84808	train-logloss:0.43927	train-au

In [5]:
# Train een mini-model met 5 bomen om de belangrijkste feature te vinden
mini_bst = xgb.train(params, dtrain, num_boost_round=5)
importance = mini_bst.get_score(importance_type="gain")
print("Belangrijkste features:", sorted(importance.items(), key=lambda x: x[1], reverse=True)[:3])


Belangrijkste features: [('type_GET', 18986.970703125), ('body_MISSING', 900.24169921875), ('runid', 146.27378845214844)]
